# Executive Dashboard: Oil & Gas (2015-2026)

Este notebook consume las vistas de negocio del esquema estrella para generar una lectura ejecutiva de desempeño operativo y productivo.

**Vistas utilizadas**
- `vw_rentabilidad_cuenca_anual`
- `vw_uptime_mensual_empresa_yacimiento`
- `vw_recuperacion_secundaria_mensual`
- `vw_water_cut_mensual_pozo`
- `vw_gor_empresa_anual`
- `vw_pareto_pozos_cuenca_detalle`
- `vw_pareto_pozos_cuenca_resumen`


In [11]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown
from sqlalchemy import create_engine

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (14, 7)


In [ ]:
# Conexion al contenedor de PostgreSQL usando variables de entorno
database_url = os.getenv('DATABASE_URL')
if not database_url:
    user = os.getenv('POSTGRES_USER', 'postgres')
    password = os.getenv('POSTGRES_PASSWORD', 'oil_pass')
    host = os.getenv('POSTGRES_HOST', 'localhost')
    port = os.getenv('POSTGRES_PORT', '5437')
    db = os.getenv('POSTGRES_DB', 'oil_gas')
    database_url = f'postgresql+psycopg://{user}:{password}@{host}:{port}/{db}'

engine = create_engine(database_url)
database_url.replace(password, '***') if 'password' in locals() else 'DATABASE_URL from env'


In [ ]:
# Carga de las 7 vistas SQL (water_cut se trae ya agregado para performance)
views_sql = {
    'rentabilidad': 'SELECT * FROM vw_rentabilidad_cuenca_anual',
    'uptime': 'SELECT * FROM vw_uptime_mensual_empresa_yacimiento',
    'rec_sec': 'SELECT * FROM vw_recuperacion_secundaria_mensual',
    'water_cut': """
        SELECT anio, cuenca, AVG(water_cut_pct) AS water_cut_pct
        FROM vw_water_cut_mensual_pozo
        WHERE anio BETWEEN 2015 AND 2026
        GROUP BY anio, cuenca
    """,
    'gor': 'SELECT * FROM vw_gor_empresa_anual',
    'pareto_detalle': 'SELECT * FROM vw_pareto_pozos_cuenca_detalle',
    'pareto_resumen': 'SELECT * FROM vw_pareto_pozos_cuenca_resumen'
}

data = {name: pd.read_sql(sql, engine) for name, sql in views_sql.items()}
for name, df in data.items():
    print(f'{name}: {df.shape[0]:,} filas x {df.shape[1]} columnas')


## 1) Pareto de Producción por Pozo

El principio de Pareto permite identificar concentración de valor: qué proporción de pozos explica la mayor parte de la producción.

**Por qué importa**: si una fracción pequeña de pozos concentra la producción, la estrategia de CAPEX/OPEX y mantenimiento debe priorizar ese subconjunto crítico.


In [ ]:
pareto = data['pareto_detalle'].copy()
cuenca_sel = pareto['cuenca'].value_counts().index[0] if not pareto.empty else None
pareto_c = pareto[pareto['cuenca'] == cuenca_sel].sort_values('orden_pozo') if cuenca_sel else pd.DataFrame()

fig = go.Figure()
if not pareto_c.empty:
    fig.add_trace(go.Bar(
        x=pareto_c['orden_pozo'],
        y=pareto_c['prod_pet_total_pozo'],
        name='Produccion por pozo',
        marker_color='#1f77b4'
    ))
    fig.add_trace(go.Scatter(
        x=pareto_c['orden_pozo'],
        y=pareto_c['pct_prod_acumulada'],
        name='% Produccion acumulada',
        yaxis='y2',
        mode='lines',
        line=dict(color='#d62728', width=3)
    ))
    fig.add_hline(y=80, line_dash='dash', line_color='gray', yref='y2')

fig.update_layout(
    title=f'Pareto Chart - Cuenca {cuenca_sel}' if cuenca_sel else 'Pareto Chart',
    xaxis_title='Ranking de pozos',
    yaxis_title='Produccion de petroleo',
    yaxis2=dict(title='% acumulado', overlaying='y', side='right', range=[0, 100]),
    legend=dict(orientation='h', y=1.1),
    template='plotly_white'
)
fig.show()


## 2) Uptime Operativo (Heatmap)

El uptime mide continuidad operativa real. Variaciones por mes y empresa suelen reflejar paradas programadas/no programadas, cuellos de botella o restricciones de superficie.

**Por qué importa**: mejorar uptime impacta directamente en producción sin necesidad inmediata de perforar nuevos pozos.


In [ ]:
uptime = data['uptime'].copy()
uptime['periodo'] = uptime['anio'].astype(str) + '-' + uptime['mes'].astype(str).str.zfill(2)
top_empresas = (
    uptime.groupby('nombre_empresa', as_index=False)['tef_total']
    .sum()
    .sort_values('tef_total', ascending=False)
    .head(15)['nombre_empresa']
)
heat_df = uptime[uptime['nombre_empresa'].isin(top_empresas)].copy()
heat_df = (
    heat_df.groupby(['nombre_empresa', 'periodo'], as_index=False)['uptime_pct']
    .mean()
)
pivot = heat_df.pivot(index='nombre_empresa', columns='periodo', values='uptime_pct').fillna(0)

plt.figure(figsize=(18, 8))
sns.heatmap(pivot, cmap='YlGnBu', linewidths=0.2, cbar_kws={'label': 'Uptime %'})
plt.title('Heatmap de Uptime Mensual por Empresa (Top 15 por TEF)')
plt.xlabel('Periodo')
plt.ylabel('Empresa')
plt.tight_layout()
plt.show()


## 3) Water Cut Promedio por Cuenca

El Water Cut es un indicador clave del estado de madurez del yacimiento. Valores crecientes suelen asociarse a agotamiento de presión y mayor costo de manejo de agua.

**Por qué importa**: cuencas con Water Cut elevado requieren rediseño de estrategia de levantamiento, inyección y tratamiento para proteger margen operativo.


In [ ]:
# Water Cut ya viene agregado desde la celda de carga para acelerar el notebook
wc_agg = data['water_cut'].copy()

fig_wc = px.line(
    wc_agg,
    x='anio',
    y='water_cut_pct',
    color='cuenca',
    markers=True,
    title='Evolucion del Water Cut Promedio por Cuenca'
)
fig_wc.update_layout(template='plotly_white', xaxis_title='Año', yaxis_title='Water Cut %')
fig_wc.show()

display(wc_agg.groupby('cuenca', as_index=False)['water_cut_pct'].mean().sort_values('water_cut_pct', ascending=False).head(10))


## Conclusiones de Negocio (5 hallazgos clave)

A continuación se generan hallazgos automáticos a partir de las vistas para soportar decisiones ejecutivas.


In [ ]:
rent = data['rentabilidad'].copy()
gor = data['gor'].copy()
pareto_res = data['pareto_resumen'].copy()

latest_year = int(rent['anio'].max()) if not rent.empty else None

top_rent = (
    rent[rent['anio'] == latest_year]
    .sort_values('ratio_rentabilidad', ascending=False)
    .head(1)
) if latest_year is not None else pd.DataFrame()

gor_mix = gor.groupby('orientacion', as_index=False)['nombre_empresa'].nunique() if not gor.empty else pd.DataFrame()

top_pareto = (
    pareto_res.sort_values('pct_prod_top_20_pozos', ascending=False).head(1)
) if not pareto_res.empty else pd.DataFrame()

wc_trend = wc_agg.groupby('anio', as_index=False)['water_cut_pct'].mean() if 'wc_agg' in locals() and not wc_agg.empty else pd.DataFrame()
wc_delta = None
if not wc_trend.empty and len(wc_trend) > 1:
    wc_delta = float(wc_trend.iloc[-1]['water_cut_pct'] - wc_trend.iloc[0]['water_cut_pct'])

uptime_rank = (
    uptime.groupby('nombre_empresa', as_index=False)['uptime_pct']
    .mean()
    .sort_values('uptime_pct', ascending=False)
    .head(1)
) if 'uptime' in locals() and not uptime.empty else pd.DataFrame()

insights = []

if not top_rent.empty:
    r = top_rent.iloc[0]
    insights.append(f"En {latest_year}, la cuenca con mayor rentabilidad por pozo fue **{r['cuenca']}** ({r['ratio_rentabilidad']:.2f} barriles por pozo activo).")

if not uptime_rank.empty:
    u = uptime_rank.iloc[0]
    insights.append(f"La empresa con mejor eficiencia operativa promedio fue **{u['nombre_empresa']}** con un uptime medio de **{u['uptime_pct']:.1f}%**.")

if wc_delta is not None:
    trend_txt = 'aumentó' if wc_delta > 0 else 'disminuyó'
    insights.append(f"El Water Cut promedio del sistema **{trend_txt} {abs(wc_delta):.2f} p.p.** en el periodo analizado, señal de cambio en madurez de reservorios.")

if not gor_mix.empty:
    mix_text = ', '.join([f"{row['orientacion']}: {int(row['nombre_empresa'])} empresas" for _, row in gor_mix.iterrows()])
    insights.append(f"La matriz productiva por GOR muestra: **{mix_text}**.")

if not top_pareto.empty:
    p = top_pareto.iloc[0]
    insights.append(f"En **{p['cuenca']}**, el top 20% de pozos explica aproximadamente **{p['pct_prod_top_20_pozos']:.1f}%** de la produccion total de petroleo.")

# Garantizar 5 hallazgos
while len(insights) < 5:
    insights.append('Se recomienda profundizar el analisis por yacimiento y tipo de reservorio para robustecer decisiones de inversion.')

display(Markdown('\n'.join([f'{i+1}. {txt}' for i, txt in enumerate(insights[:5])])))
